# Karafka — Capture Without Sending via `class_eval`

**Pattern**: wrap `WaterDrop::Producer#produce_sync` so outgoing Kafka messages can be intercepted before they leave the process.

- **Phase 1 (capture)**: `produce_sync` is intercepted — message payload recorded, original method never called, Kafka never touched.
- **Phase 2 (passthrough)**: patch is off — `produce_sync` calls through normally.

Rollback for Kafka is out of scope here (you cannot unsend a message); the capture alone is enough to inspect what *would have been* published.

## Setup — WaterDrop producer

`deliver: false` makes WaterDrop go through its full pipeline (middleware, instrumentation) but skip the actual rdkafka delivery — so no broker is needed for the demo.

In [ ]:
require 'waterdrop'

PRODUCER.close rescue nil if defined?(PRODUCER)
Object.send(:remove_const, :PRODUCER) if defined?(PRODUCER)
PRODUCER = WaterDrop::Producer.new do |config|
  config.deliver = false
  config.kafka   = { 'bootstrap.servers': 'localhost:9092' }
  config.logger  = Logger.new(File::NULL)
end

puts 'Producer ready (deliver: false -- no broker needed in demo)'

## CaptureContext

In [ ]:
module CaptureContext
  @capturing = false
  @messages  = []

  def self.start!     = (@capturing = true; @messages = [])
  def self.stop!      = (@capturing = false)
  def self.capturing? = @capturing
  def self.add(msg)   = (@messages << { topic: msg[:topic], payload: msg[:payload] })
  def self.messages   = @messages.dup
end

puts 'CaptureContext ready'

## The patch — `WaterDrop::Producer.class_eval`

We patch the real `WaterDrop::Producer` — the same class your app uses in production.  
In capture mode the message is recorded and `original_produce_sync` is **never called**, so nothing reaches Kafka regardless of the `deliver` setting.

In [ ]:
# NOTE: run this cell only once per kernel session.
# Re-running alias_method points :original_produce_sync at the already-patched method, causing infinite recursion.
# To reset, restart the kernel and re-run all cells from the top.
WaterDrop::Producer.class_eval do
  alias_method :original_produce_sync, :produce_sync

  def produce_sync(**kwargs)
    if CaptureContext.capturing?
      CaptureContext.add(kwargs)
      return
    end
    original_produce_sync(**kwargs)
  end
end

puts 'Patch applied -- WaterDrop::Producer.class_eval produce_sync wrapped'

## Demo messages

In [ ]:
Object.send(:remove_const, :EVENTS) if defined?(EVENTS)
EVENTS = [
  { topic: 'user_events', payload: { event: 'user_created', name: 'Alice' }.to_json },
  { topic: 'user_events', payload: { event: 'user_updated', name: 'Bob'   }.to_json }
]

## Phase 1 — Capture

`produce_sync` is called for each event.  
The patch intercepts every call, records the payload, and returns immediately.  
Kafka is never contacted.

In [ ]:
CaptureContext.start!

EVENTS.each { |e| PRODUCER.produce_sync(**e) }

puts "Intercepted #{CaptureContext.messages.size} message(s) -- Kafka never touched:"
CaptureContext.messages.each_with_index do |msg, i|
  puts "  [#{i}] topic=#{msg[:topic]}  payload=#{msg[:payload]}"
end

## Phase 2 — Passthrough

Capture is off.  
`produce_sync` calls through to `original_produce_sync`.  
(`deliver: false` in this demo — swap to a real broker in production.)

In [ ]:
CaptureContext.stop!

EVENTS.each { |e| PRODUCER.produce_sync(**e) }
puts "#{EVENTS.size} message(s) dispatched"


## Cleanup

Run when you are done with the notebook.

In [ ]:
PRODUCER.close
puts "Producer closed."